# 07 · Fine-tune with checkpoints (drift conditions)

LoRA fine-tunes the base model under one condition and seed, saving an adapter every few steps so 08 can watch safety drift (docs/02 §3.3). **Run once per (condition, seed)**: dev runs use seed 0 for all four conditions; test runs use seeds 1 and 2.

| | |
|---|---|
| **Kaggle settings** | Accelerator: GPU T4 (single GPU) · Internet: **On** · Secrets: `HF_TOKEN` |
| **Attach as inputs** | `safetypulse-lib` (sp_utils.py) · output of **01** |
| **Reads** | `ft_<condition>.jsonl` |
| **Writes** (`/kaggle/working/`) | `runs/<condition>-s<seed>/checkpoint-<step>/` (LoRA adapters) |

Keep this notebook and its outputs **private**: they contain harmful prompts.
After it finishes: *Save Version → Save & Run All*, then download the executed notebook into
`kaggle/executed/` and attach this notebook's output to the next notebook.

In [1]:
# Use one GPU: Trainer + DataParallel on T4 x2 is slower and less stable for this.
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

'0'

In [2]:
%pip install -q peft

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Setup: find sp_utils.py (attach the `safetypulse-lib` dataset: see kaggle/README.md)
import glob, os, sys
_found = (glob.glob("/kaggle/input/**/sp_utils.py", recursive=True)
          + glob.glob("/kaggle/usr/lib/**/sp_utils.py", recursive=True)
          + glob.glob("../lib/sp_utils.py"))
assert _found, "sp_utils.py not found. Attach the safetypulse-lib dataset (Add Input) and re-run."
sys.path.insert(0, os.path.dirname(os.path.abspath(_found[0])))
import sp_utils as sp
sp.env_info()

{'torch': '2.11.0+cu130',
 'transformers': '5.20.2',
 'cuda_available': True,
 'on_kaggle': True,
 'smoke': False,
 'peft': '0.21.2',
 'gpus': ['Tesla T4'],
 'capability': [7, 5],
 'vram_gb': 15.6}

In [4]:
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments

CONDITION = "gsm8k"      # gsm8k (D-benign-1) | alpaca (D-benign-2) | harm (D-harm) | none (D-none)
SEED = 0                 # 0 = dev run; 1, 2 = test runs
MODEL = os.environ.get("SP_MODEL", sp.DEFAULT_MODEL)
HP = {
    # batch x accum = 16 (harm: 4). Per-device batch 2 keeps Tiny Aya's 262k-vocab logits within a T4.
    "gsm8k":  dict(epochs=2, lr=2e-4, batch=2, accum=8, save_every=25),
    "alpaca": dict(epochs=2, lr=2e-4, batch=2, accum=8, save_every=25),
    "none":   dict(epochs=2, lr=2e-4, batch=2, accum=8, save_every=25),
    "harm":   dict(epochs=5, lr=2e-4, batch=2, accum=2, save_every=10),
}[CONDITION]
if sp.SMOKE:
    HP.update(epochs=1, batch=2, accum=1, save_every=1)
MAX_LEN = 512
LORA = dict(r=16, lora_alpha=32, lora_dropout=0.05,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
RUN = f"{CONDITION}-s{SEED}"
OUT = sp.out_dir(f"runs/{RUN}")
sp.set_seed(SEED)
rows = sp.load_jsonl(sp.find_input(f"ft_{CONDITION}.jsonl"))
print(RUN, len(rows), "examples", HP)

gsm8k-s0 2000 examples {'epochs': 2, 'lr': 0.0002, 'batch': 2, 'accum': 8, 'save_every': 25}


In [5]:
tok = sp.load_tokenizer(MODEL)
train = sp.sft_features(tok, [r["messages"] for r in rows], MAX_LEN)
steps = (len(train) // (HP["batch"] * HP["accum"])) * HP["epochs"]
print(f"{len(train)} features, ~{steps} optimizer steps, ~{steps // HP['save_every']} checkpoints")

if MODEL == sp.TINY_MODEL:
    model = sp._tiny_model()
else:
    # Frozen base weights: fp32 if they fit one T4 (<= ~2B params), else fp16 (Tiny Aya 3.35B). The LoRA
    # adapters peft adds stay fp32 either way, and training uses fp16 autocast.
    src = sp.model_source(MODEL)
    base_dtype = torch.float16 if sp.too_big_for_one_gpu(src, torch.float32) else torch.float32
    model = AutoModelForCausalLM.from_pretrained(src, dtype=base_dtype)
    print("base weights", base_dtype)
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
model = get_peft_model(model, LoraConfig(task_type="CAUSAL_LM", **LORA))
model.print_trainable_parameters()
model.save_pretrained(OUT / "checkpoint-0")       # untrained adapter == base model, the step-0 reference

copied CohereLabs__tiny-aya-global to local disk in 0.5 min


1986 features, ~248 optimizer steps, ~9 checkpoints


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

base weights torch.float16


ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

In [ ]:
use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
args = TrainingArguments(
    output_dir=str(OUT), seed=SEED, num_train_epochs=HP["epochs"], learning_rate=HP["lr"],
    per_device_train_batch_size=HP["batch"], gradient_accumulation_steps=HP["accum"],
    lr_scheduler_type="cosine", warmup_steps=max(1, int(0.03 * steps)), weight_decay=0.0,
    logging_steps=5, save_strategy="steps", save_steps=HP["save_every"], save_only_model=True,
    fp16=torch.cuda.is_available() and not use_bf16, bf16=use_bf16,
    report_to="none", remove_unused_columns=False, dataloader_num_workers=2,
)
trainer = Trainer(model=model, args=args, train_dataset=train, data_collator=sp.SFTCollator(tok.pad_token_id))
with sp.gpu_timer() as t:
    trainer.train()
print(f"trained in {t['seconds'] / 60:.1f} min")

In [ ]:
import matplotlib.pyplot as plt
trainer.save_model(OUT / f"checkpoint-{trainer.state.global_step}")
hist = [h for h in trainer.state.log_history if "loss" in h]
sp.save_json({"run": RUN, "condition": CONDITION, "seed": SEED, "model": MODEL, "hp": HP, "lora": LORA,
              "base_dtype": str(next(model.parameters()).dtype), "n_examples": len(train), "train_minutes": t["seconds"] / 60, "log_history": trainer.state.log_history,
              "provenance": sp.provenance(SEED)}, OUT / "train_log.json")
plt.plot([h["step"] for h in hist], [h["loss"] for h in hist])
plt.xlabel("step"); plt.ylabel("train loss"); plt.title(RUN)
sorted(p.name for p in OUT.glob("checkpoint-*"))